# Predictive Maintenance Models - Main Dataset Experiments

## Impor Required Libraries

In [ ]:
import os
import sys
import json
import textwrap
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import gaussian_kde
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle

from imblearn.over_sampling import SMOTENC

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OrdinalEncoder

## Dataset Loading

Dataset Sources:
- https://www.kaggle.com/datasets/shivamb/machine-predictive-maintenance-classification


### Load the Datasets

In [ ]:
df_predictive_maintenance = pd.read_csv('data/predictive_maintenance.csv')

### Predictive Maintenance Dataset

In [ ]:
df_predictive_maintenance.head()

## Define the Category, Numeric and Target

In [ ]:
# Predictive Maintenance Dataset
df_predictive_maintenance_id = ["UDI", "Product ID"]
df_predictive_maintenance_category = ["Type"]
df_predictive_maintenance_target = ["Target", "Failure Type"]
df_predictive_maintenance_numeric = ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]

## Auto Dataset Assessing

In [ ]:
def assess_dataframe(df, target=None, high_card_threshold=0.5):
    print("Data Assessment Report")

    n_rows, n_cols = df.shape
    print(f"\nShape of Dataset: {n_rows} Rows x {n_cols} Columns")

    print("\nColumn Information:")
    info_table = pd.DataFrame({
        "DataType": df.dtypes,
        "Non-Null Count": df.notnull().sum(),
        "Null Count": df.isnull().sum(),
        "Null %": (df.isnull().sum() / n_rows * 100).round(2),
        "Unique Values": df.nunique(dropna=True),
        "Unique %": (df.nunique(dropna=True) / n_rows * 100).round(2),
    })
    print(info_table)

    dup_count = df.duplicated().sum()
    dup_percent = round((dup_count / n_rows) * 100, 2) if n_rows else 0.0
    print(f"\nDuplicate Rows: {dup_count} ({dup_percent}%)")

    const_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    print("\nConstant Columns:")
    print(const_cols if const_cols else "None")

    cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    high_card_cols = []
    for c in cat_cols:
        ratio = (df[c].nunique(dropna=True) / n_rows) if n_rows else 0.0
        if ratio >= high_card_threshold:
            high_card_cols.append((c, round(ratio*100, 2)))
    print("\nHigh Cardinality Categorical Columns (>= {:.0f}% Unique):".format(high_card_threshold*100))
    if high_card_cols:
        for c, pct in high_card_cols:
            print(f"- {c}: {pct}% Unique")
    else:
        print("None")

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        print("\nNumeric Columns Summary:")
        modes = {}
        if len(num_cols) > 0:
            mode_df = df[num_cols].mode(dropna=True)
            for c in num_cols:
                modes[c] = mode_df[c].iloc[0] if not mode_df.empty and c in mode_df else np.nan
        summary = pd.DataFrame({
            "Mean": df[num_cols].mean(numeric_only=True),
            "Median": df[num_cols].median(numeric_only=True),
            "Mode": pd.Series(modes),
            "Min": df[num_cols].min(numeric_only=True),
            "Q1": df[num_cols].quantile(0.25, numeric_only=True),
            "Q3": df[num_cols].quantile(0.75, numeric_only=True),
            "Max": df[num_cols].max(numeric_only=True),
            "Standard Deviation": df[num_cols].std(numeric_only=True),
            "Skewness": df[num_cols].skew(numeric_only=True),
            "Kurtosis": df[num_cols].kurt(numeric_only=True)
        }).round(4)
        print(summary)

        print("\nOutlier Counts (IQR method):")
        outlier_rows = []
        for c in num_cols:
            series = df[c].dropna()
            if series.empty:
                outlier_rows.append((c, 0, 0.0))
                continue
            q1 = series.quantile(0.25)
            q3 = series.quantile(0.75)
            iqr = q3 - q1
            if iqr == 0:
                outlier_rows.append((c, 0, 0.0))
                continue
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            count = ((series < lower) | (series > upper)).sum()
            pct = round(count / len(series) * 100, 2)
            outlier_rows.append((c, int(count), pct))
        if outlier_rows:
            outlier_table = pd.DataFrame(outlier_rows, columns=["Column", "Outliers", "Outliers %"])
            print(outlier_table)

        print("\nPearson Correlation (Numeric):")
        pearson_corr = df[num_cols].corr(method="pearson")
        print(pearson_corr.round(3))

        print("\nSpearman Correlation (Numeric):")
        spearman_corr = df[num_cols].corr(method="spearman")
        print(spearman_corr.round(3))
    else:
        print("\nNo Numeric columns found.")

    print("\nColumns > 50% Missing Values:")
    high_missing = info_table[info_table["Null %"] > 50].sort_values("Null %", ascending=False)
    if not high_missing.empty:
        print(high_missing[["Null Count", "Null %"]])
    else:
        print("None")

    print("\nMemory Usage:")
    total_mem = df.memory_usage(deep=True).sum()
    print(f"Total: {total_mem} bytes ({round(total_mem/1024,2)} KB, {round(total_mem/(1024*1024),2)} MB)")
    memory_per_col = df.memory_usage(deep=True)
    memory_table = pd.DataFrame({"Bytes": memory_per_col}).sort_values("Bytes", ascending=False)
    print("\nPer Column Memory:")
    print(memory_table)

    if target is not None and target in df.columns:
        print(f"\nClass Balance for '{target}':")
        vc = df[target].value_counts(dropna=False)
        vp = df[target].value_counts(normalize=True, dropna=False).mul(100).round(2)
        bal = pd.DataFrame({"Count": vc, "Percent %": vp})
        print(bal)

    print("\nData Assessment Completed")

## Expl(ora/ana)tory Data Analysis - Predictive Maintenance Dataset

### Assessing Dataset with Different Target

In [ ]:
assess_dataframe(df_predictive_maintenance, target=df_predictive_maintenance_target[0])


In [ ]:
assess_dataframe(df_predictive_maintenance, target=df_predictive_maintenance_target[1])


### Histogram & KDE of Numeric Features

In [ ]:
plt.figure(figsize=(15, 10))

for i, col in enumerate(df_predictive_maintenance_numeric, 1):
    plt.subplot(2, 3, i)
    
    plt.hist(df_predictive_maintenance[col], bins=30, density=True, color='steelblue', alpha=0.7, edgecolor='white')

    kde = gaussian_kde(df_predictive_maintenance[col].dropna())
    x_vals = np.linspace(df_predictive_maintenance[col].min(), df_predictive_maintenance[col].max(), 200)
    plt.plot(x_vals, kde(x_vals), color="gray", linewidth=2)
    
    plt.xlabel(col)
    plt.ylabel("Density")
    plt.title(col)
    
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.tight_layout()
plt.show()

### Distribution of Category - `Type`

In [ ]:
rows, cols = 2, 4
fig = plt.figure(figsize=(16, 9))
gs = GridSpec(rows, cols, figure=fig, wspace=0.6, hspace=0.9)

for idx, col in enumerate(df_predictive_maintenance_category):
    r, c = divmod(idx, cols)

    cell = gs[r, c].subgridspec(
        2, 2,
        height_ratios=[12, 1],
        width_ratios=[3.0, 3.4]
    )
    ax_pie   = fig.add_subplot(cell[0, 0])
    ax_leg   = fig.add_subplot(cell[0, 1])
    ax_title = fig.add_subplot(cell[1, :])

    vals = df_predictive_maintenance[col].value_counts()
    labels = vals.index.to_list()
    sizes  = vals.values
    total  = sizes.sum()

    cmap   = plt.get_cmap("Blues")
    colors = cmap(np.linspace(0.45, 0.85, len(vals)))

    ax_pie.pie(
        sizes, startangle=90, colors=colors,
        wedgeprops=dict(width=0.38), labels=None
    )
    ax_pie.set_aspect('equal')
    ax_pie.set_xticks([]); ax_pie.set_yticks([])

    ax_leg.set_xlim(0, 1); ax_leg.set_ylim(0, 1)
    ax_leg.axis('off')

    n = len(labels)
    y_top = 0.80
    dy = min(0.16, 0.9 / (n + 1))

    for i, (lab, cnt, color) in enumerate(zip(labels, sizes, colors)):
        y = y_top - i * dy

        ax_leg.add_patch(Rectangle((0.05, y - 0.025), 0.05, 0.05, facecolor=color, edgecolor='none'))

        ax_leg.text(0.12, y, f"{lab}: {cnt} ({cnt/total:.1%})",
                    va='center', ha='left', fontsize=9)

    ax_title.axis('off')
    ax_title.text(0.5, 0.5, f"Distribution of {col}",
                  ha='center', va='center', fontsize=12, fontweight='bold')

plt.show()

### Distribution of Target 1 -  `Target`

In [ ]:
rows, cols = 2, 4
fig = plt.figure(figsize=(16, 9))
gs = GridSpec(rows, cols, figure=fig, wspace=0.6, hspace=0.9)

for idx, col in enumerate(df_predictive_maintenance_target[0:1]):
    r, c = divmod(idx, cols)

    cell = gs[r, c].subgridspec(
        2, 2,
        height_ratios=[12, 1],
        width_ratios=[3.0, 3.4]
    )
    ax_pie   = fig.add_subplot(cell[0, 0])
    ax_leg   = fig.add_subplot(cell[0, 1])
    ax_title = fig.add_subplot(cell[1, :])

    vals = df_predictive_maintenance[col].value_counts()
    labels = vals.index.to_list()
    sizes  = vals.values
    total  = sizes.sum()

    cmap   = plt.get_cmap("Blues")
    colors = cmap(np.linspace(0.45, 0.85, len(vals)))

    ax_pie.pie(
        sizes, startangle=90, colors=colors,
        wedgeprops=dict(width=0.38), labels=None
    )
    ax_pie.set_aspect('equal')
    ax_pie.set_xticks([]); ax_pie.set_yticks([])

    ax_leg.set_xlim(0, 1); ax_leg.set_ylim(0, 1)
    ax_leg.axis('off')

    n = len(labels)
    y_top = 0.80
    dy = min(0.16, 0.9 / (n + 1))

    for i, (lab, cnt, color) in enumerate(zip(labels, sizes, colors)):
        y = y_top - i * dy

        ax_leg.add_patch(Rectangle((0.05, y - 0.025), 0.05, 0.05, facecolor=color, edgecolor='none'))

        ax_leg.text(0.12, y, f"{lab}: {cnt} ({cnt/total:.1%})",
                    va='center', ha='left', fontsize=9)

    ax_title.axis('off')
    ax_title.text(0.5, 0.5, f"Distribution of {col}",
                  ha='center', va='center', fontsize=12, fontweight='bold')

plt.show()

### Distribution of Target 2 - `Failure Type`

In [ ]:
rows, cols = 2, 4
fig = plt.figure(figsize=(16, 9))
gs = GridSpec(rows, cols, figure=fig, wspace=0.6, hspace=0.9)

for idx, col in enumerate(df_predictive_maintenance_target[1:2]):
    r, c = divmod(idx, cols)

    cell = gs[r, c].subgridspec(
        2, 2,
        height_ratios=[12, 1],
        width_ratios=[3.0, 3.4]
    )
    ax_pie   = fig.add_subplot(cell[0, 0])
    ax_leg   = fig.add_subplot(cell[0, 1])
    ax_title = fig.add_subplot(cell[1, :])

    vals = df_predictive_maintenance[col].value_counts()
    labels = vals.index.to_list()
    sizes  = vals.values
    total  = sizes.sum()

    cmap   = plt.get_cmap("Blues")
    colors = cmap(np.linspace(0.45, 0.85, len(vals)))

    ax_pie.pie(
        sizes, startangle=90, colors=colors,
        wedgeprops=dict(width=0.38), labels=None
    )
    ax_pie.set_aspect('equal')
    ax_pie.set_xticks([]); ax_pie.set_yticks([])

    ax_leg.set_xlim(0, 1); ax_leg.set_ylim(0, 1)
    ax_leg.axis('off')

    n = len(labels)
    y_top = 0.80
    dy = min(0.16, 0.9 / (n + 1))

    for i, (lab, cnt, color) in enumerate(zip(labels, sizes, colors)):
        y = y_top - i * dy

        ax_leg.add_patch(Rectangle((0.05, y - 0.025), 0.05, 0.05, facecolor=color, edgecolor='none'))

        ax_leg.text(0.12, y, f"{lab}: {cnt} ({cnt/total:.1%})",
                    va='center', ha='left', fontsize=9)

    ax_title.axis('off')
    ax_title.text(0.5, 0.5, f"Distribution of {col}",
                  ha='center', va='center', fontsize=12, fontweight='bold')

plt.show()

### Correlation Matrix - Target 1

In [ ]:
plt.figure(figsize=(8, 6))

corr_matrix = df_predictive_maintenance[df_predictive_maintenance_numeric + [df_predictive_maintenance_target[0]]].corr()

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="Blues",
    fmt=".2f",
    linewidths=0.5,
    vmin=-1, vmax=1,
    cbar_kws={"shrink": 0.8, "label": "Correlation"}
)

plt.title("Correlation Matrix (Numeric + Target 1)", fontsize=13, pad=15, fontweight="bold")
plt.xticks(rotation=90, ha='center')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Data Preprocessing

### Change Columns Name

In [ ]:
column_rename_map = {
    # Identifiers
    "UDI": "unit_id",
    "Product ID": "product_id",
    # Categorical
    "Type": "engine_type",
    
    # Targets
    "Target": "target",
    "Failure Type": "failure_type",
    
    # Numeric
    "Air temperature [K]": "air_temp",
    "Process temperature [K]": "process_temp",
    "Rotational speed [rpm]": "rpm",
    "Torque [Nm]": "torque_nm",
    "Tool wear [min]": "tool_wear",
}

df_predictive_maintenance = df_predictive_maintenance.rename(columns=column_rename_map)
df_predictive_maintenance.head()

### Defined Constant & Variables

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Choose scaler: "standard" or "minmax"
SCALER_TYPE = "standard" # set to "minmax" if you prefer MinMaxScaler

# Split ratio
TEST_SIZE = 0.20

# Timestamp synthesis cadence
TS_STEP_MINUTES = 10

OUT_DIR = Path("processed"); OUT_DIR.mkdir(exist_ok=True, parents=True)

NUMERIC_COLS = ["air_temp", "process_temp", "rpm", "torque_nm", "tool_wear"]
CAT_COLS = ["engine_type"]

def log_info(msg): 
    print(f"[INFO] {msg}")
    
def log_res(msg):  
    print(f"[RESULT] {msg}")
    
def get_scaler():
    return StandardScaler() if SCALER_TYPE == "standard" else MinMaxScaler()

### Preprocessing Function

In [ ]:
def fit_transform_cat_for_smotenc(df_cat: pd.DataFrame):
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    arr = enc.fit_transform(df_cat)
    df_enc = pd.DataFrame(arr, columns=df_cat.columns, index=df_cat.index)
    return enc, df_enc

def transform_cat(enc: OrdinalEncoder, df_cat: pd.DataFrame):
    arr = enc.transform(df_cat)
    return pd.DataFrame(arr, columns=df_cat.columns, index=df_cat.index)

def smotenc_resample(X_mix: pd.DataFrame, y: pd.Series, cat_cols: list, random_state=SEED):
    cat_idx = [X_mix.columns.get_loc(c) for c in cat_cols]
    sm = SMOTENC(categorical_features=cat_idx, random_state=random_state)
    X_res, y_res = sm.fit_resample(X_mix.values, y.values)
    return X_res, y_res

def add_synthetic_timestamps(df_like: pd.DataFrame, group_col="unit_id", start=None, step_minutes=TS_STEP_MINUTES):
    if start is None:
        start = datetime.now()
    dfx = df_like.sort_values([group_col]).reset_index(drop=True)
    dfx["timestamp"] = dfx.groupby(group_col).cumcount().apply(
        lambda k: start + timedelta(minutes=step_minutes*k)
    )
    return dfx

def assign_unit_ids_like(dfx_len: int, unit_pool: np.ndarray):
    if len(unit_pool) == 0:
        # Fallback single unit if pool is empty
        unit_pool = np.array([0])
    return np.random.choice(unit_pool, size=dfx_len, replace=True)

def pack_and_save(
    X_arr, y_arr, enc: OrdinalEncoder, scaler, base_df_pool: pd.DataFrame,
    out_path: Path, note: str
):
    # Reconstruct a human-readable DataFrame:
    # X_arr columns order = CAT_COLS + NUMERIC_COLS
    cols = CAT_COLS + NUMERIC_COLS
    tmp = pd.DataFrame(X_arr, columns=cols)

    # Inverse transform categorical to original labels
    tmp_cat = tmp[CAT_COLS].round().clip(lower=-1).astype(int)  # defensive cast
    tmp[CAT_COLS] = enc.inverse_transform(tmp_cat.values)

    # Attach unit_id sampled from the pool (to keep per-unit group structure plausible)
    unit_pool = base_df_pool["unit_id"].values
    tmp["unit_id"] = assign_unit_ids_like(len(tmp), unit_pool)

    # Attach product_id sampled from pool as well (optional, for richer schema)
    if "product_id" in base_df_pool.columns:
        prod_pool = base_df_pool["product_id"].values
        tmp["product_id"] = np.random.choice(prod_pool, size=len(tmp), replace=True)

    # Attach y
    tmp["target"] = y_arr.astype(int)

    # Timestamp synthesis (applied outside when needed; here we only save if already present)
    # Save
    tmp = tmp[["unit_id", "product_id"] + CAT_COLS + NUMERIC_COLS + ["target"] + ([ "timestamp"] if "timestamp" in tmp.columns else [])]
    tmp.to_csv(out_path, index=False)
    log_res(f"Saved {note}: {out_path.name} ({len(tmp)} rows)")
    return tmp

### Anti-Leakage Preprocessing Pipeline

In [ ]:
# 1) Split first
X_all = df_predictive_maintenance[CAT_COLS + NUMERIC_COLS].copy()
y_all = df_predictive_maintenance["target"].copy()

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, stratify=y_all, random_state=SEED
)
log_info(f"[A] Split done. Train: {X_train_df.shape}, Test: {X_test_df.shape}")

# 2) Normalize numeric (fit on train)
scaler_A = get_scaler()
X_train_num = pd.DataFrame(
    scaler_A.fit_transform(X_train_df[NUMERIC_COLS]),
    columns=NUMERIC_COLS, index=X_train_df.index
)
X_test_num = pd.DataFrame(
    scaler_A.transform(X_test_df[NUMERIC_COLS]),
    columns=NUMERIC_COLS, index=X_test_df.index
)

# 3) Encode categorical for SMOTENC
enc_A, X_train_cat_enc = fit_transform_cat_for_smotenc(X_train_df[CAT_COLS])
X_test_cat_enc = transform_cat(enc_A, X_test_df[CAT_COLS])

# 4) Build mixed matrices
X_train_mix = pd.concat([X_train_cat_enc, X_train_num], axis=1)
X_test_mix  = pd.concat([X_test_cat_enc,  X_test_num], axis=1)

# 5) Balance only the training set
X_tr_res_A, y_tr_res_A = smotenc_resample(X_train_mix, y_train, CAT_COLS, random_state=SEED)

# 6) Timestamp synthesis after resampling
# Rebuild train frame with unit_id/product_id from the TRAIN pool
train_pool = df_predictive_maintenance.loc[X_train_df.index, ["unit_id", "product_id"]].copy()
test_pool  = df_predictive_maintenance.loc[X_test_df.index,  ["unit_id", "product_id"]].copy()

train_path_A = OUT_DIR / "anti-leakage-dataset_train.csv"
test_path_A  = OUT_DIR / "anti-leakage-dataset_test.csv"

train_df_A = pack_and_save(
    X_tr_res_A, y_tr_res_A, enc_A, scaler_A, train_pool, train_path_A, note="[A] Train"
)
test_df_A = pack_and_save(
    X_test_mix.values, y_test.values, enc_A, scaler_A, test_pool, test_path_A, note="[A] Test"
)

# Add timestamps per unit_id
train_df_A = add_synthetic_timestamps(train_df_A, group_col="unit_id", step_minutes=TS_STEP_MINUTES)
test_df_A  = add_synthetic_timestamps(test_df_A,  group_col="unit_id", step_minutes=TS_STEP_MINUTES)

# Overwrite with timestamped versions
train_df_A.to_csv(train_path_A, index=False)
test_df_A.to_csv(test_path_A, index=False)

log_res(f"[A] Final saved: {train_path_A.name}, {test_path_A.name}")
display(train_df_A.head())
display(test_df_A.head())


### Sythesize Preprocessing Pipeline

In [ ]:
# 1) Normalize numeric on the FULL dataset (intentional leakage for this design)
scaler_B = get_scaler()
X_num_B = pd.DataFrame(
    scaler_B.fit_transform(df_predictive_maintenance[NUMERIC_COLS]),
    columns=NUMERIC_COLS, index=df_predictive_maintenance.index
)

# 2) Encode categorical on FULL dataset
enc_B, X_cat_enc_B = fit_transform_cat_for_smotenc(df_predictive_maintenance[CAT_COLS])

# 3) Combine, then SMOTENC on FULL dataset
X_mix_B = pd.concat([X_cat_enc_B, X_num_B], axis=1)
y_B = df_predictive_maintenance["target"].copy()

X_full_res_B, y_full_res_B = smotenc_resample(X_mix_B, y_B, CAT_COLS, random_state=SEED)
log_info(f"[B] After SMOTENC full: {X_full_res_B.shape}")

# 4) Timestamp synthesis on the resampled FULL set
# Build frame and assign unit_id/product_id sampled from the full pool
full_pool = df_predictive_maintenance[["unit_id", "product_id"]].copy()
full_path_tmp = OUT_DIR / "_tmp_synthesize_full.csv"
full_df_B = pack_and_save(
    X_full_res_B, y_full_res_B, enc_B, scaler_B, full_pool, full_path_tmp, note="[B] Full (tmp)"
)
# Add timestamps
full_df_B = add_synthetic_timestamps(full_df_B, group_col="unit_id", step_minutes=TS_STEP_MINUTES)

# 5) Now split AFTER normalization+balancing+timestamps
X_B = full_df_B[["unit_id", "product_id"] + CAT_COLS + NUMERIC_COLS + ["timestamp"]].copy()
y_B = full_df_B["target"].copy()

X_train_B, X_test_B, y_train_B, y_test_B = train_test_split(
    X_B, y_B, test_size=TEST_SIZE, stratify=y_B, random_state=SEED
)

# 6) Save final train/test for synthesize-first design
train_path_B = OUT_DIR / "synthesize-dataset_train.csv"
test_path_B  = OUT_DIR / "synthesize-dataset_test.csv"

train_df_B = pd.concat([X_train_B, y_train_B], axis=1)
test_df_B  = pd.concat([X_test_B,  y_test_B], axis=1)

train_df_B.to_csv(train_path_B, index=False)
test_df_B.to_csv(test_path_B, index=False)

# Clean temp
try:
    full_path_tmp.unlink(missing_ok=True)
except Exception:
    pass

log_res(f"[B] Final saved: {train_path_B.name}, {test_path_B.name}")
display(train_df_B.head())
display(test_df_B.head())

In [ ]:
from collections import OrderedDict

summary = OrderedDict({
    "Anti-Leakage (train)": OUT_DIR / "anti-leakage-dataset_train.csv",
    "Anti-Leakage (test)":  OUT_DIR / "anti-leakage-dataset_test.csv",
    "Synthesize (train)":   OUT_DIR / "synthesize-dataset_train.csv",
    "Synthesize (test)":    OUT_DIR / "synthesize-dataset_test.csv",
})

for k, p in summary.items():
    print(f"{k}: {p} | exists={p.exists()} | bytes={p.stat().st_size if p.exists() else 'NA'}")

# Quick schema check
anti_leakage = pd.read_csv("processed/anti-leakage-dataset_train.csv")
synthesize_balanced = pd.read_csv("processed/synthesize-dataset_train.csv")

print("\nAnti-Leakage Dataset:")
display(anti_leakage.head())
print("\nSynthesize Dataset:")
display(synthesize_balanced.head())